# Chapter 14 — Regressions: Did the Model Change, or the Measurement?

**Book alignment:** PyTorch From First Principles, Chapter 14

**Question this notebook isolates:** Is an observed validation-loss delta a real model change, or is it explained by seed noise (baseline spread) or by a changed evaluation path (identical weights, different number)?

In [ ]:
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__)

## 1 — Noise floor: same seed repeats, different seeds spread

One thread plus a fixed seed gives bit-identical reruns here; twelve... six seeds map the spread that chance alone produces.

In [ ]:
def gen_data(n_train=600, n_val=200, noise=0.30, data_seed=0):
    g = torch.Generator().manual_seed(data_seed)
    W = torch.randn(32, 4, generator=g)
    Xtr = torch.randn(n_train, 32, generator=g)
    Xva = torch.randn(n_val, 32, generator=g)
    ytr = (Xtr @ W).argmax(-1)
    yva = (Xva @ W).argmax(-1)
    flip = torch.rand(n_train, generator=g) < noise
    ytr[flip] = torch.randint(0, 4, (int(flip.sum()),), generator=g)
    return (Xtr, ytr), (Xva, yva)

(Xtr, ytr), (Xva, yva) = gen_data()

def train_run(seed, lr=0.02):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, 4))
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=3e-3)
    m.train()
    for ep in range(8):
        perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(seed + ep))
        for i in range(0, len(Xtr), 128):
            idx = perm[i:i + 128]
            opt.zero_grad()
            F.cross_entropy(m(Xtr[idx]), ytr[idx]).backward()
            opt.step()
    m.eval()
    with torch.no_grad():
        vl = float(F.cross_entropy(m(Xva), yva))
    return vl

a1, a2 = train_run(0), train_run(0)
sweep = [train_run(s) for s in range(6)]
import statistics
med = statistics.median(sweep)
sd = statistics.pstdev(sweep)
print(f'same seed twice: {a1:.4f} vs {a2:.4f} identical={a1 == a2}')
print(f'sweep: {[round(v, 4) for v in sweep]} median={med:.4f} stdev={sd:.4f} spread={max(sweep) - min(sweep):.4f}')

In [ ]:
assert a1 == a2
assert (max(sweep) - min(sweep)) > 0
assert sd > 0
print('noise floor verified')

## 2 — Paired comparison: a real shift dwarfs the noise floor

Same seeds on both sides (common random numbers). A 10x learning rate is a genuine model change: every paired delta goes one way and the shift is many noise-floors wide.

In [ ]:
cand = [train_run(s, lr=0.2) for s in range(6)]
import statistics as st
deltas = [c - b for b, c in zip(sweep, cand)]
shift = st.median(cand) - med
print(f'baseline median={med:.4f} candidate median={st.median(cand):.4f} shift={shift:.4f}')
print('paired deltas:', [round(d, 4) for d in deltas])
print(f'worse on {sum(d > 0 for d in deltas)}/6 seeds; shift/noise-sd={shift / (sd + 1e-12):.1f}x')

In [ ]:
assert all(d > 0 for d in deltas)
assert shift > 5 * sd
print('paired regression verified')

## 3 — Measurement change: identical weights, different numbers

Fix one trained model; vary only the eval path. Mode, subset, input scale, and loss reduction each move the metric with zero model change. A fingerprint over the eval contract catches it.

In [ ]:
torch.manual_seed(0)
m = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 4))
m.eval()

def ev(mode_train, scale, idx, reduction='mean'):
    m.train(mode_train)
    with torch.no_grad():
        return float(F.cross_entropy(m(Xva[idx] * scale), yva[idx], reduction=reduction))

idx_all = torch.arange(len(Xva))
idx_half = torch.arange(0, len(Xva), 2)
canon = ev(False, 1.0, idx_all)
v_trainmode = ev(True, 1.0, idx_all)
v_subset = ev(False, 1.0, idx_half)
v_scaled = ev(False, 1.15, idx_all)
v_sum = ev(False, 1.0, idx_all, reduction='sum')
print(f'canonical {canon:.4f} train-mode {v_trainmode:.4f} subset {v_subset:.4f} scaled {v_scaled:.4f} sum {v_sum:.1f}')

def fingerprint(mode_train, scale, n_idx, reduction):
    s = f'{mode_train}|{scale}|{n_idx}|{reduction}'
    return hashlib.sha256(s.encode()).hexdigest()[:12]

fp_c = fingerprint(False, 1.0, len(idx_all), 'mean')
fp_s = fingerprint(False, 1.15, len(idx_all), 'mean')
print('fingerprints:', fp_c, fp_s, 'same:', fp_c == fp_s)

In [ ]:
assert fp_c != fp_s
assert abs(v_scaled - canon) > 1e-4
assert abs(v_subset - canon) > 1e-4
assert v_sum > 10 * canon
print('measurement-change verified')

## What we earned

A moved metric is a symptom with three explanations: seed noise, changed measurement, or changed system. Map the baseline spread first, pair seeds when valid, fingerprint the eval path — and only then explain the mechanism.

Chapter 15 assembles everything: a tiny interrogable language model checked by every instrument in the book.